### Tweaked Code From: [Week 7](https://git.arts.ac.uk/mtanska/24-25-bsc-cc-2-big-data/blob/main/week_07_text_to_emotion_to_colour.ipynb)

In [114]:
import json

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import re
import spacy
ln_model = spacy.load("en_core_web_lg")


# then partial imports
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from scipy import spatial
from random import randint
from transformers import (
    pipeline,
    RobertaForSequenceClassification,
    RobertaTokenizerFast,
)

In [115]:
from huggingface_hub import login
HUGGINGFACE_TOKEN = "token"
login(token=HUGGINGFACE_TOKEN)

In [116]:
XKCD_COLOR = "data/xkcd.json"
TEXT_FILE = "YOUR_TEXT_FILE_LOCATION"

OUTPUT_JSON = "emo_colours.json"

In [117]:
themes = ['Love and Relationships',
'Nature and the Envrionment',
'Loss and Grief',
'Resilience and Strength',
'Hope and Optimism',
'Joy and Celebration',
'Beauty and Aesthetics',
'Time and Transience',
'Spirituality and Wonder']

In [118]:
emotion_pipeline = pipeline("text-classification", model="arpanghoshal/EmoRoBERTa", tokenizer="arpanghoshal/EmoRoBERTa")
theme_emotions = []
for theme in themes:
    result = emotion_pipeline(theme)[0]
    emotion = result["label"].lower()
    theme_emotions.append({
        "theme": theme,
        "emotion": emotion
    })
# chatgpt- needed the "theme": and "emtion:" to make it a dict
print(theme_emotions)

All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at arpanghoshal/EmoRoBERTa.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.
Device set to use 0


[{'theme': 'Love and Relationships', 'emotion': 'love'}, {'theme': 'Nature and the Envrionment', 'emotion': 'neutral'}, {'theme': 'Loss and Grief', 'emotion': 'neutral'}, {'theme': 'Resilience and Strength', 'emotion': 'neutral'}, {'theme': 'Hope and Optimism', 'emotion': 'optimism'}, {'theme': 'Joy and Celebration', 'emotion': 'joy'}, {'theme': 'Beauty and Aesthetics', 'emotion': 'neutral'}, {'theme': 'Time and Transience', 'emotion': 'neutral'}, {'theme': 'Spirituality and Wonder', 'emotion': 'excitement'}]


In [119]:
color_data = json.loads(open(XKCD_COLOR).read())

In [120]:
def vec(text):
    return ln_model(text).vector

In [121]:
colour_vectors = {name: vec(name) for name in colors.keys()}

In [122]:
def eucl_dist(vec_1, vec_2):
    return np.linalg.norm(vec_1 - vec_2)

In [123]:
def get_closest_colour(text_data, colour_vectors):
    
    text_vec = vec(text_data)
    eucl_distances = {colour_name: eucl_dist(text_vec, colour_vec) for colour_name, colour_vec in colour_vectors.items()}
    similarity = pd.DataFrame({'euclidean_distance': eucl_distances})
    
    # sorting by euclidean distance as cosine similarity was often returning the same colours for different emotions
    closest_colour = similarity.sort_values('euclidean_distance').iloc[0].name
    
    return closest_colour

In [124]:
closest_colours = [
    {
        "theme": item["theme"],
        "emotion": item["emotion"],
        "color": get_closest_colour(item["emotion"], colour_vectors),
        "hex": colors[get_closest_colour(item["emotion"], colour_vectors)]
    }
    for item in theme_emotions
]
# chatgpt helped- i wanted to have it layed out/print it so i could see the hex code instead of rgb

In [126]:
for item in closest_colours:
    print(f"{item['theme']}: {item['emotion']}  {item['color']} ({item['hex']})")
    # chatgpt helped- i wanted to have it layed out/print it so i could see the hex code instead of rgb

Love and Relationships: love  really light blue (#d4ffff)
Nature and the Envrionment: neutral  strong blue (#0c06f7)
Loss and Grief: neutral  strong blue (#0c06f7)
Resilience and Strength: neutral  strong blue (#0c06f7)
Hope and Optimism: optimism  greyblue (#77a1b5)
Joy and Celebration: joy  blue with a hint of purple (#533cc6)
Beauty and Aesthetics: neutral  strong blue (#0c06f7)
Time and Transience: neutral  strong blue (#0c06f7)
Spirituality and Wonder: excitement  blue with a hint of purple (#533cc6)
